# CTCF + NT preprocess demo

This notebook demonstrates the TFBS preprocessing logic for CTCF: raw ChIP-seq peaks -> centered positive windows -> GC-matched negative samples -> balanced dataset preview. It uses a temporary output directory when the full pipeline is enabled, so no processed CSV is saved into the Exp2 project tree.

In [1]:
from pathlib import Path
import os
import sys

EXP2_ROOT = Path("/dataset/zjn_zjj/DLM/GLM_Benchmark/Exp2_Motif-Interpretability-Analysis")
SRC_ROOT = EXP2_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

LEGACY_MOTIF_ROOT = Path("/dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif")
PEAKS_DIR = LEGACY_MOTIF_ROOT / "Data" / "original_peaks"
REFERENCE_DIR = PEAKS_DIR / "reference"
TF_NAME = "CTCF"
MODEL_TYPE = "NT"

print(f"EXP2_ROOT = {EXP2_ROOT}")
print(f"PEAKS_DIR = {PEAKS_DIR}")
print(f"REFERENCE_DIR = {REFERENCE_DIR}")

EXP2_ROOT = /dataset/zjn_zjj/DLM/GLM_Benchmark/Exp2_Motif-Interpretability-Analysis
PEAKS_DIR = /dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/Data/original_peaks
REFERENCE_DIR = /dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/Data/original_peaks/reference


In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

peak_file = PEAKS_DIR / f"{TF_NAME}_peaks.bed"
chrom_sizes = REFERENCE_DIR / "hg38.chrom.sizes"
blacklist = REFERENCE_DIR / "hg38-blacklist.v2.bed"
reference_fa = REFERENCE_DIR / "hg38.fa"

for path in [peak_file, chrom_sizes, blacklist, reference_fa]:
    print(f"{path}: {'OK' if path.exists() else 'MISSING'}")

peaks = pd.read_csv(peak_file, sep="\t", header=None, names=["chrom", "start", "end", "peak_id", "score"])
print(f"Raw CTCF peaks: {len(peaks):,}")
peaks.head()

/dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/Data/original_peaks/CTCF_peaks.bed: OK
/dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/Data/original_peaks/reference/hg38.chrom.sizes: OK
/dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/Data/original_peaks/reference/hg38-blacklist.v2.bed: OK
/dataset/zjn_zjj/DLM/10_21_previous_work/Data_associated_with_Graduation/motif_discovery/motif/Data/original_peaks/reference/hg38.fa: OK
Raw CTCF peaks: 43,387


,chrom,start,end,peak_id,score
0,chr1,181910,182072,peak1,20.56064
1,chr1,267886,268126,peak2,25.05717
2,chr1,586059,586247,peak3,25.05717
3,chr1,629819,630065,peak4,38.53932
4,chr1,633895,634163,peak5,88.62894


In [3]:
# Lightweight preview of positive windows without writing processed data.
WINDOW_SIZE = 1001
preview_peaks = peaks.sort_values("score", ascending=False).copy()
preview_peaks["center"] = ((preview_peaks["start"] + preview_peaks["end"]) // 2).astype(int)
preview_peaks["window_start"] = preview_peaks["center"] - WINDOW_SIZE // 2
preview_peaks["window_end"] = preview_peaks["center"] + WINDOW_SIZE // 2 + 1
preview_peaks = preview_peaks[preview_peaks["window_start"] >= 0].copy()

status = pd.DataFrame({
    "stage": ["raw_peaks", "valid_centered_windows"],
    "count": [len(peaks), len(preview_peaks)],
})
status

,stage,count
0,raw_peaks,43387
1,valid_centered_windows,43383


In [7]:
# Full reproduction path. Keep disabled until you want to call bedtools and generate temporary intermediates.
RUN_FULL_PREPROCESS = False
TOTAL_RANDOM_REGIONS = 50_000
NUM_PROCESSES = 4

if RUN_FULL_PREPROCESS:
    import tempfile
    from exp2_attention.preprocess.build_tfbs_dataset import (
        setup_config,
        create_filtered_chrom_sizes,
        filter_blacklist,
        process_positive_samples,
        generate_random_regions,
        parallel_nuc_calculation,
        filter_regions,
        extract_sequences,
        calculate_gc_content,
        build_balanced_dataset,
    )

    with tempfile.TemporaryDirectory(prefix="exp2_ctcf_preprocess_") as tmpdir:
        cfg = setup_config(
            TF_NAME,
            str(peak_file),
            tmpdir,
            reference_dir=REFERENCE_DIR,
            total_samples=TOTAL_RANDOM_REGIONS,
            num_processes=NUM_PROCESSES,
        )
        os.makedirs(cfg["TF_DIR"], exist_ok=True)
        create_filtered_chrom_sizes(cfg)
        filter_blacklist(cfg)
        positive_df = process_positive_samples(cfg)
        generate_random_regions(cfg)
        parallel_nuc_calculation(cfg)
        filter_regions(cfg)
        neg_sequences = extract_sequences(cfg)
        neg_seq_df = calculate_gc_content(neg_sequences)
        balanced_df = build_balanced_dataset(positive_df, neg_seq_df, cfg)
else:
    positive_df = pd.DataFrame(columns=["sequence", "label", "gc_content"])
    neg_seq_df = pd.DataFrame(columns=["sequence", "label", "gc_content"])
    balanced_df = pd.DataFrame(columns=["sequence", "label"])
    print("RUN_FULL_PREPROCESS=False: enable it to reproduce balanced data construction in a temporary directory.")

RUN_FULL_PREPROCESS=False: enable it to reproduce balanced data construction in a temporary directory.


In [8]:
if not balanced_df.empty:
    display(pd.DataFrame({
        "split_candidate": ["balanced_dataset"],
        "total": [len(balanced_df)],
        "positive": [int((balanced_df["label"] == 1).sum())],
        "negative": [int((balanced_df["label"] == 0).sum())],
        "positive_ratio": [float((balanced_df["label"] == 1).mean())],
    }))
    display(balanced_df.head())
else:
    print("Balanced dataset preview will appear here after RUN_FULL_PREPROCESS=True.")

Balanced dataset preview will appear here after RUN_FULL_PREPROCESS=True.


In [9]:
if not positive_df.empty and not neg_seq_df.empty:
    gc_plot_df = pd.concat([
        positive_df[["gc_content"]].assign(label="positive_peak"),
        neg_seq_df[["gc_content"]].assign(label="candidate_negative"),
    ], ignore_index=True)
    ax = sns.boxplot(data=gc_plot_df, x="label", y="gc_content")
    ax.set_title("CTCF GC content before final balancing")
    ax.set_xlabel("")
    ax.set_ylabel("GC content")
else:
    print("GC boxplot will appear here after RUN_FULL_PREPROCESS=True.")

GC boxplot will appear here after RUN_FULL_PREPROCESS=True.
